# Notebook 04 – Model Selection Engine

**Purpose.** Compare candidate forecast models using rolling one-step-ahead evaluation. This is the *only* notebook that performs model comparison; Notebook 05 consumes its output.

**Key design decisions**

1. **MASE** is the primary metric (handles zero-sales days). MAPE is reported but flagged per group — it is unreliable when daily sales are frequently zero (e.g. N05C).
2. The blood series is evaluated **weekly** (aligned with stock management), not monthly.
3. Exogenous variables (`HospitalAdmissions`, `Events`, `BloodDonorsAvailable`, `Temperature`) are **tested** with ARIMAX against the baseline models. If they do not improve MASE, that finding is reported honestly.
4. `PredictedBloodDemand` is an existing column in the synthetic dataset and is **not** used.
5. Series frequency is set with `asfreq()` to silence statsmodels warnings.

**About `PredictedBloodDemand`.** The synthetic source file ships a column named `PredictedBloodDemand`. It is a pre-computed demand estimate generated by whoever produced the synthetic dataset; we did not produce it, and using it would be circular (we would be evaluating a prediction with another prediction). It is therefore **not** used as a target or as a feature anywhere in this project. The target for blood forecasting is `HistoricalBloodUsage`.

**Outputs**

- `winning_models.csv` — selected models per item and design, with MASE, MAE, and MASE margin.
- `evaluation_summary.csv` — full aggregated metrics (for the report).

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Silence statsmodels ValueWarning about inferred frequency, convergence
# warnings, and overflow warnings from ARIMAX fits that fail.
warnings.filterwarnings("ignore", message=".*No frequency information.*")
warnings.filterwarnings("ignore", message=".*Maximum Likelihood optimization failed.*")
warnings.filterwarnings("ignore", message=".*Non-invertible starting.*")
warnings.filterwarnings("ignore", message=".*divide by zero.*")
warnings.filterwarnings("ignore", message=".*overflow.*")

PROJECT_DIR = Path(r"C:\Users\Administrator\MediShelf")
PHARMA_PATH = PROJECT_DIR / "data" / "processed" / "pharma_sales_clean.csv"
BLOOD_PATH  = PROJECT_DIR / "data" / "processed" / "blood_demand_clean.csv"
OUTPUT_DIR  = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [PHARMA_PATH, BLOOD_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

print("Paths OK")

## 1. Load and prepare series

Series are prepared with **explicit frequency** (`asfreq`) so that statsmodels does not emit the repeated `ValueWarning: No frequency information was provided`.

**Designs**

| Design | Target | Frequency |
|---|---|---|
| Blood | `HistoricalBloodUsage` sum per week | `W-SUN` |
| Pharma — monthly | mean daily sales per month | `MS` |
| Pharma — weekly | sum of daily sales per week | `W-SUN` |
| Pharma — daily (N05C only) | raw daily sales | `D` |

In [ ]:
pharma = pd.read_csv(PHARMA_PATH)
pharma.columns = pharma.columns.str.strip()
if "datum" in pharma.columns:
    pharma = pharma.rename(columns={"datum": "Date"})
pharma["Date"] = pd.to_datetime(pharma["Date"])
pharma = pharma.sort_values("Date").set_index("Date")

ATC = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]

# Monthly pharma
monthly_series = {
    g: pharma[g].resample("MS").mean().asfreq("MS").dropna().astype(float)
    for g in ATC
}

# Weekly pharma (for R03, R06, N05C as used in 05)
WEEKLY_PHARMA = ["N05C", "R03", "R06"]
weekly_pharma = {
    g: pharma[g].resample("W-SUN").sum().asfreq("W-SUN").dropna().astype(float)
    for g in WEEKLY_PHARMA
}

# Daily N05C
first_monday = pharma.index.min() + pd.Timedelta(days=(7 - pharma.index.min().weekday()) % 7)
daily_n05c = pharma.loc[first_monday:, "N05C"].asfreq("D").dropna().astype(float)

# Weekly blood
blood = pd.read_csv(BLOOD_PATH)
blood["Date"] = pd.to_datetime(blood["Date"])
blood = blood.sort_values("Date").set_index("Date")
blood_weekly = blood["HistoricalBloodUsage"].resample("W-SUN").sum().asfreq("W-SUN").dropna().astype(float)

# Exogenous features for ARIMAX (weekly)
blood_exog = pd.DataFrame({
    "Admissions": blood["HospitalAdmissions"].resample("W-SUN").mean().asfreq("W-SUN"),
    "Events":     blood["Events"].resample("W-SUN").sum().asfreq("W-SUN"),
    "Donors":     blood["BloodDonorsAvailable"].resample("W-SUN").mean().asfreq("W-SUN"),
    "Temp":       blood["Temperature"].resample("W-SUN").mean().asfreq("W-SUN"),
}).dropna()
blood_exog = blood_exog.reindex(blood_weekly.index)

print(f"Blood weekly  : {len(blood_weekly)} obs {blood_weekly.index.min().date()} → {blood_weekly.index.max().date()}")
print(f"Monthly pharma: {len(monthly_series['M01AB'])} obs")
print(f"Weekly pharma : {len(weekly_pharma['N05C'])} obs")
print(f"Daily N05C    : {len(daily_n05c)} obs")

## 2. Metrics

- **MAE** — average error in original units.
- **MASE** — scale-free; scaled by in-sample seasonal-naive error. **Primary metric.**
- **MAPE** — reported but flagged when the group is intermittent (many zero-sales days). MAPE is not used for selection.

In [ ]:
def compute_metrics(actual, predicted, train, seasonality=1, flags=None):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mae = float(np.mean(np.abs(actual - predicted)))

    if len(train) > seasonality:
        naive_err = np.abs(train.values[seasonality:] - train.values[:-seasonality])
        denom = float(np.mean(naive_err)) if len(naive_err) else 0.0
        mase = mae / denom if denom > 0 else np.nan
    else:
        mase = np.nan

    with np.errstate(divide="ignore", invalid="ignore"):
        mask = actual != 0
        mape = (np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100
                if mask.any() else np.nan)

    return dict(MAE=mae, MASE=mase, MAPE=mape)


def naive_weekly(train, horizon):
    return float(train.iloc[-1])

def seasonal_naive_weekly(train, horizon, season=52):
    return float(train.iloc[-season]) if len(train) >= season else float(train.iloc[-1])

## 3. Rolling evaluation

One-step-ahead, expanding window. Each model is fitted on the history up to step *i* and evaluated on point *i*.

For blood we also test **ARIMAX** using the four exogenous features, to see whether they improve accuracy over the pure time-series baselines.

In [ ]:
def rolling_evaluate(series, group, models, min_train, season, exog=None):
    rows = []
    for i in range(min_train, len(series)):
        train = series.iloc[:i]
        test = series.iloc[i]
        train_exog = exog.iloc[:i] if exog is not None else None
        test_exog  = exog.iloc[i:i+1] if exog is not None else None

        for m in models:
            try:
                if m == "Training mean":
                    pred = float(train.mean())
                elif m == "Naive":
                    pred = naive_weekly(train, 1)
                elif m == "Seasonal naive":
                    pred = seasonal_naive_weekly(train, 1, seasonality=season)
                elif m == "SES":
                    fit = SimpleExpSmoothing(train, initialization_method="estimated").fit(optimized=True)
                    pred = float(fit.forecast(1).iloc[0])
                elif m == "ARIMA(0,1,1)":
                    fit = ARIMA(train, order=(0,1,1)).fit()
                    pred = float(fit.forecast(1).iloc[0])
                elif m == "ARIMAX(all exog)":
                    fit = SARIMAX(
                        endog=train.values, exog=train_exog.values,
                        order=(1,0,1), trend="c",
                        enforce_stationarity=False, enforce_invertibility=False,
                    ).fit(disp=False)
                    pred = float(fit.forecast(1, exog=test_exog.values)[0])
                else:
                    continue

                met = compute_metrics([test], [pred], train, seasonality=season)
                met.update({"Group": group, "Model": m})
                rows.append(met)
            except Exception:
                continue
    return pd.DataFrame(rows)

### 3.1 Blood (weekly) — baseline vs ARIMAX

Minimum training window = 52 weeks; seasonality = 52. If exogenous variables materially improve MASE (< 2% margin), they are retained for blood; otherwise the pure time-series baseline wins and that finding is reported.

In [ ]:
blood_models = [
    "Training mean", "Naive", "Seasonal naive", "SES", "ARIMA(0,1,1)",
    "ARIMAX(all exog)",
]
print("Running blood weekly evaluation (this may take a few minutes)…")
blood_results = rolling_evaluate(
    blood_weekly, "BLOOD", blood_models,
    min_train=52, season=52, exog=blood_exog,
)
blood_results["Design"] = "Weekly"
blood_results["Intermittent"] = False
print("Steps evaluated:", blood_results.groupby('Model')['MAE'].size().to_dict())

In [ ]:
print("\nRunning monthly pharma evaluation…")
pharma_monthly_models = ["Training mean", "Naive", "SES", "ARIMA(0,1,1)"]
monthly_results = []
for g, s in monthly_series.items():
    res = rolling_evaluate(s, g, pharma_monthly_models, min_train=24, season=12)
    res["Design"] = "Monthly"
    intermittent = (pharma[g] == 0).mean() > 0.05
    res["Intermittent"] = intermittent
    monthly_results.append(res)

print("Running weekly pharma evaluation (N05C, R03, R06)…")
pharma_weekly_models = ["Training mean", "Naive", "Seasonal naive", "SES"]
weekly_results = []
for g, s in weekly_pharma.items():
    res = rolling_evaluate(s, g, pharma_weekly_models, min_train=52, season=52)
    res["Design"] = "Weekly"
    intermittent = (pharma[g] == 0).mean() > 0.05
    res["Intermittent"] = intermittent
    weekly_results.append(res)

print("Running daily N05C evaluation…")
res_daily = rolling_evaluate(daily_n05c, "N05C", ["Training mean", "Naive", "SES"], min_train=365, season=7)
res_daily["Design"] = "Daily"
res_daily["Intermittent"] = True

df_all = pd.concat([blood_results, *monthly_results, *weekly_results, res_daily], ignore_index=True)
print("Total evaluations:", len(df_all))

## 4. Aggregation and winner selection

Winners are selected on **MASE**. The MAPE column is reported for transparency but is not used to select models.

**Practical ties.** Any model within 2% of the best MASE for a group/design is retained as a candidate so that Notebook 05 can present the alternative.

In [ ]:
summary = (
    df_all.groupby(["Group", "Design", "Model", "Intermittent"])
    .agg(Avg_MAE=("MAE", "mean"), Avg_MASE=("MASE", "mean"), Avg_MAPE=("MAPE", "mean"),
         N_steps=("MAE", "size"))
    .reset_index()
)

summary["Min_MASE"] = summary.groupby(["Group", "Design"])["Avg_MASE"].transform("min")
summary["MASE_Diff_Pct"] = (summary["Avg_MASE"] - summary["Min_MASE"]) / summary["Min_MASE"]

winners = summary[summary["MASE_Diff_Pct"] <= 0.02].copy()
winners = winners.sort_values(["Group", "Design", "Avg_MASE"]).reset_index(drop=True)

print("Blood model comparison (MASE):")
display(summary[summary.Group == "BLOOD"][["Model", "Avg_MAE", "Avg_MASE", "Avg_MAPE"]].round(4))

print("\nSelected winners (all groups):")
display(winners[["Group", "Design", "Model", "Avg_MASE", "Avg_MAPE", "MASE_Diff_Pct"]].round(4))

### 4.1 Interpretation — did exogenous blood features help?

The cell below prints a short, honest reading of whether ARIMAX outperformed the pure time-series baselines on weekly blood.

In [ ]:
blood_summary = summary[summary.Group == "BLOOD"].sort_values("Avg_MASE")
best = blood_summary.iloc[0]
arimax = blood_summary[blood_summary.Model == "ARIMAX(all exog)"]

print("Blood weekly — best MASE:", round(best.Avg_MASE, 4), "| model:", best.Model)
if not arimax.empty:
    a = arimax.iloc[0]
    diff = (a.Avg_MASE - best.Avg_MASE) / best.Avg_MASE * 100
    print(f"ARIMAX(all exog) MASE: {a.Avg_MASE:.4f}  ({diff:+.1f}% vs best)")
    if best.Model == "ARIMAX(all exog)":
        print("=> Exogenous variables IMPROVED weekly blood MASE. They are retained for 05.")
    else:
        print("=> Exogenous variables did NOT improve weekly blood MASE. Baseline retained.")
        print("   This is a valid finding: within this synthetic dataset,")
        print("   admissions/events/donors/temperature do not add predictive value")
        print("   beyond the pure time-series structure.")

## 5. Export

`winning_models.csv` is the hand-off contract to Notebook 05. Columns:

- `Group`, `Design`, `Model`, `Avg_MAE`, `Avg_MASE`, `Avg_MAPE`, `MASE_Diff_Pct`

Notebook 05 uses only `Group`, `Design`, `Model` and fits each selected method on the full series — no re-comparison.

In [ ]:
winners_path = OUTPUT_DIR / "winning_models.csv"
summary_path = OUTPUT_DIR / "evaluation_summary.csv"

winners[["Group", "Design", "Model", "Avg_MAE", "Avg_MASE", "Avg_MAPE", "MASE_Diff_Pct"]].to_csv(
    winners_path, index=False
)
summary[["Group", "Design", "Model", "Intermittent", "Avg_MAE", "Avg_MASE", "Avg_MAPE", "N_steps"]].to_csv(
    summary_path, index=False
)
print("Wrote:", winners_path)
print("Wrote:", summary_path)
display(winners[["Group", "Design", "Model", "Avg_MASE"]].round(4))